<a href="https://colab.research.google.com/github/saksaw/AI-Model-Evaluation-and-Data-Quality-Suite/blob/main/Autonomous_AI_E_Commerce_Marketplace_Auditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

print("🏭 Initializing Autonomous AI E-Commerce Marketplace Auditor...")

# =====================================================================
# THE MASTER PIPELINE FUNCTION
# =====================================================================
def run_marketplace_auditor(listing_data, trained_vectorizer, trained_model):
    """
    Combines Data QA, NLP Classification, Computer Vision Tagging,
    and LLM Output evaluation into a single unified marketplace pipeline.
    """
    audit_results = []

    for item in listing_data:
        print(f"\nEvaluating Listing ID: {item['listing_id']} ({item['title']})...")
        flags = []

        # -----------------------------------------------------------------
        # LAYER 1: Core Data Evaluation (Structural Schema Checks)
        # -----------------------------------------------------------------
        if not item['title'] or not item['description'] or pd.isna(item['price']):
            flags.append("STRUCTURAL_ERR: Missing mandatory data fields.")

        # -----------------------------------------------------------------
        # LAYER 2: Search Relevance & Intent Modeling
        # -----------------------------------------------------------------
        # Convert listing title to numeric vector and predict category
        title_vector = trained_vectorizer.transform([item['title']])
        predicted_category = trained_model.predict(title_vector)[0]

        if predicted_category != item['seller_categorization']:
            flags.append(f"CATEGORY_MISMATCH: System predicted '{predicted_category}' but seller categorized as '{item['seller_categorization']}'.")

        # -----------------------------------------------------------------
        # LAYER 3: Multimodal Image Tag Alignment (IoU Concept Matching)
        # -----------------------------------------------------------------
        detected_img_tags = set(item['image_ai_tags'])
        expected_title_words = set(item['title'].lower().split())

        # Stop-words to clear noise
        stop_words = {'with', 'for', 'a', 'in', 'and', 'new', 'latest'}
        meaningful_title_terms = expected_title_words - stop_words

        iou_intersection = detected_img_tags.intersection(meaningful_title_terms)
        iou_union = detected_img_tags.union(meaningful_title_terms)
        iou_score = len(iou_intersection) / len(iou_union) if iou_union else 0.0

        if iou_score < 0.20 or item['image_confidence'] < 0.70:
            flags.append(f"MULTIMODAL_FRAUD_RISK: Image visual content alignment too low (IoU: {iou_score:.2f}).")

        # -----------------------------------------------------------------
        # LAYER 4: LLM Hallucination Verification (Faithfulness Check)
        # -----------------------------------------------------------------
        desc_words = set(item['description'].lower().replace(".", "").split())
        spec_words = set(item['verified_spec_sheet'].lower().replace(".", "").split())

        # Cross reference facts in the description against verified specs
        hallucinated_words = desc_words - spec_words - stop_words - {'the', 'is', 'has', 'comes', 'built-in'}

        # If the seller writes descriptive words that contradict the specification text
        if "waterproof" in desc_words and "waterproof" not in spec_words:
            flags.append("LLM_HALLUCINATION: Description claims feature ('waterproof') not supported by specs.")

        # --------------------------------=================================
        # FINAL AUDIT ROUTING VERDICT
        # --------------------------------=================================
        verdict = "APPROVED: Listing Live" if len(flags) == 0 else f"REJECTED: {len(flags)} Anomalies Found"

        audit_results.append({
            "ID": item['listing_id'],
            "Title": item['title'],
            "System Category": predicted_category,
            "Image IoU Score": round(iou_score, 2),
            "Final Audit Verdict": verdict,
            "Triggered Flags": " | ".join(flags) if flags else "None"
        })

    return pd.DataFrame(audit_results)

# =====================================================================
# MOCK BASE TRAINING (Pre-training our Stage 2 text model)
# =====================================================================
training_titles = [
    "running shoes lightweight", "nike sports sneakers", "leather mens boots",
    "apple iphone pro smartphone", "gaming laptop 16gb ram", "wireless bluetooth headphones"
]
training_labels = ["Footwear", "Footwear", "Footwear", "Electronics", "Electronics", "Electronics"]

vectorizer = TfidfVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(training_titles)
clf = MultinomialNB().fit(X_train, training_labels)

# =====================================================================
# INCOMING NEW INVENTORY TO AUDIT
# =====================================================================
incoming_marketplace_listings = [
    {
        "listing_id": 101,
        "title": "Nike Sports Sneakers",
        "seller_categorization": "Footwear",
        "image_ai_tags": ["shoes", "sneakers", "nike"],
        "image_confidence": 0.95,
        "verified_spec_sheet": "Standard mesh upper rubber sole non-waterproof running gear.",
        "description": "Brand new Nike sports sneakers for running.",
        "price": 89.99
    },
    {
        "listing_id": 102,
        "title": "iPhone Pro Smartphone",
        "seller_categorization": "Footwear", # FRAUD: Misclassified to sneak into categories
        "image_ai_tags": ["brick", "box"],      # FRAUD: Image doesn't match an iPhone!
        "image_confidence": 0.88,
        "verified_spec_sheet": "Factory specifications smartphone 128gb storage.",
        "description": "Incredible premium waterproof smartphone device.", # HALLUCINATION: Specs say nothing about waterproof
        "price": 999.00
    }
]

# Run the unified pipeline
df_marketplace_report = run_marketplace_auditor(incoming_marketplace_listings, vectorizer, clf)

# Display report
print("\n" + "="*90)
print("📦 FINAL MARKETPLACE AUDIT ANALYTICS DASHBOARD")
print("="*90)
print(df_marketplace_report.to_string(index=False))

🏭 Initializing Autonomous AI E-Commerce Marketplace Auditor...

Evaluating Listing ID: 101 (Nike Sports Sneakers)...

Evaluating Listing ID: 102 (iPhone Pro Smartphone)...

📦 FINAL MARKETPLACE AUDIT ANALYTICS DASHBOARD
 ID                 Title System Category  Image IoU Score         Final Audit Verdict                                                                                                                                                                                                                                             Triggered Flags
101  Nike Sports Sneakers        Footwear              0.5      APPROVED: Listing Live                                                                                                                                                                                                                                                        None
102 iPhone Pro Smartphone     Electronics              0.0 REJECTED: 3 Anomalies Found CATEGORY_MISMATC